In [ ]:
# 01 · 단계 ACT CONFIG · 기존 ACT/DP와 별도 결과
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_act_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 2000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 30,
    'team': 'my-team',

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


In [ ]:
# 06 · EASY · 짧은 동작 확인 · 100 iteration + 2회 실행, 별도 임시 모델
experiment.smoke("easy")


In [ ]:
# 07 · EASY · 복구 시연 수집 · 성공 에피소드만 저장, 중단하면 완료 시점부터 재개
experiment.collect("easy")


In [ ]:
# 08 · EASY · 단계 ACT 학습 · 기존 데이터 + 복구 시연, 체크포인트 자동 업로드
experiment.train("easy")


In [ ]:
# 09 · EASY · 8회 테스트 + 영상 + 별도 2회 단계/행동 기록
experiment.test("easy")
experiment.diagnose("easy")


In [ ]:
# 10 · EASY · 설정 비교 + 최종 100회 평가 · 평가 한도 200스텝 유지
experiment.evaluate("easy")


In [ ]:
# 11 · MEDIUM · 짧은 동작 확인 · 100 iteration + 2회 실행, 별도 임시 모델
experiment.smoke("medium")


In [ ]:
# 12 · MEDIUM · 복구 시연 수집 · 성공 에피소드만 저장, 중단하면 완료 시점부터 재개
experiment.collect("medium")


In [ ]:
# 13 · MEDIUM · 단계 ACT 학습 · 기존 데이터 + 복구 시연, 체크포인트 자동 업로드
experiment.train("medium")


In [ ]:
# 14 · MEDIUM · 8회 테스트 + 영상 + 별도 2회 단계/행동 기록
experiment.test("medium")
experiment.diagnose("medium")


In [ ]:
# 15 · MEDIUM · 설정 비교 + 최종 100회 평가 · 평가 한도 200스텝 유지
experiment.evaluate("medium")


In [ ]:
# 16 · HARD · 짧은 동작 확인 · 100 iteration + 2회 실행, 별도 임시 모델
experiment.smoke("hard")


In [ ]:
# 17 · HARD · 복구 시연 수집 · 성공 에피소드만 저장, 중단하면 완료 시점부터 재개
experiment.collect("hard")


In [ ]:
# 18 · HARD · 단계 ACT 학습 · 기존 데이터 + 복구 시연, 체크포인트 자동 업로드
experiment.train("hard")


In [ ]:
# 19 · HARD · 8회 테스트 + 영상 + 별도 2회 단계/행동 기록
experiment.test("hard")
experiment.diagnose("hard")


In [ ]:
# 20 · HARD · 설정 비교 + 최종 100회 평가 · 평가 한도 200스텝 유지
experiment.evaluate("hard")


In [ ]:
# 21 · 저장 결과와 영상
experiment.show_results()


In [ ]:
# 22 · 학습된 정책 패키징 · 수집 expert 제외
experiment.package()
